# 14.3 · 平滑法 / Smoothing Methods (MA, EWMA, Holt-Winters)

> **课程定位 / Where this fits**
> 第 3 课，**Part 14 · 时间序列**。最简单、最实用的一类预测方法。
> Lesson 3, **Part 14 · Time Series**. The simplest, most practical forecasting family.
>
> 不需要复杂模型, 很多预测可以靠**对过去值做(加权)平均**来完成——这就是**平滑法**。从最朴素的**移动平均(MA)**, 到给近期更高权重的**指数加权平均(EWMA)**, 再到能同时建模**水平+趋势+季节**的 **Holt-Winters 三重指数平滑**。它们简单、快、可解释, 是预测的**强基线**(很多生产系统就用它)。本课从零理解每种平滑, 并用 Holt-Winters **真正做一次预测**(按时间切分、在未来数据上评估)。
> Many forecasts need no complex model — just a **(weighted) average of past values**: **smoothing**. From the naive **moving average (MA)**, to **exponentially weighted moving average (EWMA)** that weights recent points more, to **Holt-Winters triple exponential smoothing** that models **level + trend + seasonality** together. Simple, fast, interpretable — a **strong baseline** (used in production). We understand each from scratch and **actually forecast** with Holt-Winters (chronological split, evaluated on future data).
>
> 💼 **实战/面试视角**："MA vs EWMA / 指数平滑的α是什么 / Holt-Winters 三个成分 / 加法vs乘法季节" 是预测岗常考。
> 💼 **Practical/interview angle:** "MA vs EWMA / what is α / Holt-Winters' three components / additive vs multiplicative seasonality" — common.

> 📐 **符号约定 / Notation**
> - $\alpha$ —— 平滑系数(0~1), 越大越重视近期 / smoothing factor; larger = more weight on recent
> - 水平/趋势/季节 —— Holt-Winters 的三个平滑成分 / level/trend/season

> 💡 **面试相关 / Interview-relevant**
> - "移动平均 vs 指数加权平均的区别"（出镜率 ★★★★）
> - "指数平滑的 α 含义/怎么影响平滑"（★★★★）
> - "Holt-Winters 的三重指数平滑(水平/趋势/季节)"（★★★★★）
> - "什么时候平滑法就够了(强基线)"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解移动平均(MA)与指数加权平均(EWMA)的区别。
   Understand MA vs EWMA.
2. 理解指数平滑的 α 与"递归加权"思想。
   Understand the smoothing factor α and recursive weighting.
3. 掌握 **Holt-Winters** 同时建模水平/趋势/季节。
   Master Holt-Winters modeling level/trend/season.
4. 用 Holt-Winters 做预测并**按时间正确评估**。
   Forecast with Holt-Winters and evaluate correctly (chronological).

## 目录 / TOC
1. [移动平均 MA ⭐](#1)
2. [指数加权平均 EWMA ⭐](#2)
3. [Holt-Winters 三重指数平滑 ⭐](#3)
4. [预测与评估 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 移动平均 MA ⭐ / Moving Average

**移动平均(Moving Average, MA)**: 用**最近 $k$ 个值的平均**作为当前的平滑值(或下一步的预测)。$k$ 叫窗口大小。
**Moving Average (MA):** use the **average of the last $k$ values** as the smoothed value (or next-step forecast). $k$ is the window size.

直觉: 把短期噪声"抹平", 露出底层走势。$k$ 越大→越平滑但越**滞后**(反应慢, 拐点跟不上);$k$ 越小→越贴合但越noisy。
Intuition: smooths out short-term noise to reveal the underlying movement. Larger $k$ → smoother but more **lagged** (slow to react at turns); smaller $k$ → closer-fitting but noisier.

局限(面试): ①所有过去值**权重相同**(3天前和昨天一样重要, 不合理);②**滞后**;③**预测时只会输出常数**(用最后k个的均值, 无法外推趋势/季节)。所以它更多用于**平滑/可视化**, 而非精准预测。
Limits (interview): ① all past values get **equal weight** (3 days ago = yesterday, unreasonable); ② **lag**; ③ as a forecast it's just a **constant** (the last-k mean, can't extrapolate trend/season). So MA is more for **smoothing/visualization** than accurate forecasting.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ap = [112,118,132,129,121,135,148,148,136,119,104,118, 115,126,141,135,125,149,170,170,158,133,114,140,
      145,150,178,163,172,178,199,199,184,162,146,166, 171,180,193,181,183,218,230,242,209,191,172,194,
      196,196,236,235,229,243,264,272,237,211,180,201, 204,188,235,227,234,264,302,293,259,229,203,229,
      242,233,267,269,270,315,364,347,312,274,237,278, 284,277,317,313,318,374,413,405,355,306,271,306,
      315,301,356,348,355,422,465,467,404,347,305,336, 340,318,362,348,363,435,491,505,404,359,310,337,
      360,342,406,396,420,472,548,559,463,407,362,405, 417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range("1949-01", periods=len(ap), freq="MS"); ts = pd.Series(ap, index=idx, name="passengers")

fig, ax = plt.subplots(figsize=(11, 4)); ts.plot(ax=ax, label="原始", alpha=0.5)
for k in [3, 12]:
    ts.rolling(k).mean().plot(ax=ax, label=f"{k}月移动平均")   # rolling mean = 移动平均 / moving average
ax.legend(); ax.set_title("移动平均: 窗口越大越平滑但越滞后(12月窗口抹平了季节, 露出趋势)")
plt.tight_layout(); plt.show()
print("MA: 取最近k个值的平均; k大=平滑但滞后, k小=贴合但noisy")
print("局限: 所有过去值权重相同 + 滞后 + 预测只能输出常数 → 多用于平滑可视化, 而非精准预测")


<a id="2"></a>
## 2. 指数加权平均 EWMA ⭐ / Exponentially Weighted Moving Average

移动平均的"所有过去值同等重要"不合理——直觉上**越近的值应该越重要**。**指数加权平均(EWMA)** 修正它: 权重**随时间指数衰减**, 越久远权重越小。递归形式很优雅:
MA's "all past values equally important" is unreasonable — intuitively **recent values should matter more**. **EWMA** fixes this: weights **decay exponentially**, older = smaller. The recursive form is elegant:

$$\hat{y}_t = \alpha\, y_t + (1-\alpha)\, \hat{y}_{t-1}$$

新的平滑值 = $\alpha$×(当前观测) + $(1-\alpha)$×(上一个平滑值)。**$\alpha \in (0,1)$ 是平滑系数**:$\alpha$ 大→重视近期、反应快但毛糙;$\alpha$ 小→更平滑、更滞后。展开看, 它对历史的权重是 $\alpha, \alpha(1-\alpha), \alpha(1-\alpha)^2, \dots$ ——指数衰减。
The new smoothed value = $\alpha$×(current obs) + $(1-\alpha)$×(previous smoothed). **$\alpha \in (0,1)$ is the smoothing factor:** large $\alpha$ → weights recent, fast but rough; small $\alpha$ → smoother, laggier. Unrolled, the weights on history are $\alpha, \alpha(1-\alpha), \alpha(1-\alpha)^2, \dots$ — exponential decay.

这就是**简单指数平滑(SES)** 的核心。它只建模**水平(level)**, 适合无趋势无季节的序列。下面对比不同 $\alpha$。
This is the core of **Simple Exponential Smoothing (SES)**. It models only the **level**, suited to series without trend/season. Let's compare different $\alpha$.


In [ ]:
# 不同 α 的 EWMA / EWMA with different alphas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ts.plot(ax=axes[0], label="原始", alpha=0.4, color="gray")
for alpha in [0.1, 0.5, 0.9]:
    ts.ewm(alpha=alpha).mean().plot(ax=axes[0], label=f"EWMA α={alpha}")   # 指数加权平均 / EWMA
axes[0].legend(); axes[0].set_title("EWMA: α大→反应快/贴合, α小→平滑/滞后")
# 可视化指数衰减的权重 / visualize the exponentially decaying weights
for alpha in [0.1, 0.5, 0.9]:
    w = [alpha*(1-alpha)**i for i in range(15)]
    axes[1].plot(w, "o-", label=f"α={alpha}")
axes[1].set_xlabel("往前第几期(0=当前)"); axes[1].set_ylabel("权重"); axes[1].legend()
axes[1].set_title("EWMA 的权重: 越久远权重越小(指数衰减); α越大衰减越快")
plt.tight_layout(); plt.show()
print("EWMA: ŷ_t = α·y_t + (1-α)·ŷ_{t-1}; 对历史的权重 α,α(1-α),α(1-α)²...指数衰减")
print("α控制'记忆': 大α只看近期(灵敏), 小α记得久(平滑); 这是简单指数平滑(SES)的核心, 但只建模水平")


<a id="3"></a>
## 3. Holt-Winters 三重指数平滑 ⭐ / Holt-Winters

SES 只能平滑**水平**, 处理不了趋势和季节。**Holt-Winters(三重指数平滑)** 把指数平滑扩展到**三个成分**, 各用一个平滑方程(面试核心)：
SES only smooths the **level**, missing trend and seasonality. **Holt-Winters (triple exponential smoothing)** extends it to **three components**, each with its own smoothing equation (interview core):
- **水平(level)**:序列当前的基准值(平滑系数 $\alpha$)。
  **Level:** the current baseline (smoothing $\alpha$).
- **趋势(trend)**:水平的变化速度(平滑系数 $\beta$)——让预测能**外推上升/下降**。
  **Trend:** the rate of level change (smoothing $\beta$) — lets forecasts **extrapolate up/down**.
- **季节(season)**:周期性的偏移(平滑系数 $\gamma$)——让预测能**重现季节波峰**。
  **Seasonality:** the periodic offset (smoothing $\gamma$) — lets forecasts **reproduce seasonal peaks**.

预测 = 把这三者**组合并外推**。季节也分**加法/乘法**(和 14.2 同理)——AirPassengers 用**乘法季节**。Holt-Winters 简单、无需平稳化、对趋势+季节数据效果很好, 是极常用的实务方法。
The forecast **combines and extrapolates** all three. Seasonality is also **additive/multiplicative** (like 14.2) — AirPassengers uses **multiplicative**. Holt-Winters is simple, needs no stationarization, works well on trend+seasonal data, and is a very common practical method.


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
# Holt-Winters: 趋势=加法, 季节=乘法(因季节幅度随水平增大), 周期12 / additive trend, multiplicative seasonality
hw = ExponentialSmoothing(ts, trend="add", seasonal="mul", seasonal_periods=12).fit()
fitted = hw.fittedvalues
fig, ax = plt.subplots(figsize=(11, 4))
ts.plot(ax=ax, label="原始", alpha=0.6); fitted.plot(ax=ax, label="Holt-Winters 拟合", color="red")
ax.legend(); ax.set_title("Holt-Winters 三重指数平滑: 同时拟合水平+趋势+季节(几乎重合原序列)")
plt.tight_layout(); plt.show()
print(f"学到的平滑系数: α(水平)={hw.params['smoothing_level']:.2f}, "
      f"β(趋势)={hw.params['smoothing_trend']:.2f}, γ(季节)={hw.params['smoothing_seasonal']:.2f}")
print("Holt-Winters 把指数平滑扩展到 水平/趋势/季节 三个成分, 各自递归平滑, 组合外推 → 能预测带趋势+季节的序列")


<a id="4"></a>
## 4. 预测与评估 + 小结 ⭐ / Forecasting & Evaluation

真正预测要**按时间切分**:用**前面的数据**训练, 预测**后面没见过的**, 再和真实值比(**绝不能用未来训练**——时序铁律)。我们用前 10 年训练、预测最后 2 年, 看 Holt-Winters 的预测有多准。
Real forecasting needs a **chronological split:** train on **earlier** data, predict the **later unseen** part, compare to truth (**never train on the future** — the TS golden rule). We train on the first 10 years and forecast the last 2, checking accuracy.


In [ ]:
# 按时间切分: 前120个月训练, 后24个月测试(绝不打乱!) / chronological split: first 120 train, last 24 test
train, test = ts[:120], ts[120:]
model = ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12).fit()
forecast = model.forecast(len(test))                      # 预测未来24个月 / forecast 24 months ahead

mape = (np.abs((test.values - forecast.values) / test.values)).mean() * 100   # 平均绝对百分比误差 / MAPE
fig, ax = plt.subplots(figsize=(11, 4))
train.plot(ax=ax, label="训练(前10年)")
test.plot(ax=ax, label="真实(后2年)", color="green")
forecast.plot(ax=ax, label="Holt-Winters 预测", color="red", ls="--")
ax.legend(); ax.set_title(f"Holt-Winters 预测后24个月: MAPE={mape:.1f}% (成功捕捉趋势上升+季节波峰)")
plt.tight_layout(); plt.show()
print(f"预测 MAPE = {mape:.1f}%  (平均绝对百分比误差; 越低越好)")
print("Holt-Winters 仅靠指数平滑就把趋势+季节预测得相当准 → 简单方法也是强基线")
print("注意: 训练/测试严格按时间切(前训后测), 绝不打乱; 这是时序评估的铁律(否则未来泄漏, 评估虚高)")


```
平滑法: 用过去值的(加权)平均做平滑/预测; 简单/快/可解释, 是强基线
移动平均MA: 最近k个的平均; k大平滑但滞后; 所有过去值同权(不合理); 预测只输出常数
EWMA(指数加权): ŷ_t=α·y_t+(1-α)·ŷ_{t-1}; 权重指数衰减(近期更重); α大灵敏小α平滑; =简单指数平滑SES(只建模水平)
Holt-Winters(三重指数平滑): 水平(α)+趋势(β)+季节(γ)三成分各自平滑; 能预测趋势+季节; 季节分加法/乘法
预测评估: 按时间切(前训后测), 绝不打乱; 常用MAPE/RMSE
何时够用: 有清晰趋势/季节、要快速可解释基线时, 平滑法常常足够好
```

### 💡 面试速查 / Interview cheat-sheet
1. **MA vs EWMA**: MA等权平均(滞后), EWMA指数衰减权重(近期更重)。
   MA vs EWMA: MA equal weights (laggy), EWMA exponentially decaying (recent matters more).
2. **α(平滑系数)**: 大=灵敏反应快, 小=平滑滞后; 控制"记忆长短"。
   α: large = responsive, small = smooth/laggy; controls "memory."
3. **Holt-Winters**: 水平+趋势+季节三重指数平滑; 季节有加法/乘法。
   Holt-Winters: triple exponential smoothing (level+trend+season); additive/multiplicative season.
4. **评估**: 按时间切(前训后测)绝不打乱; 用MAPE/RMSE。
   Evaluation: chronological split, never shuffle; MAPE/RMSE.
5. **强基线**: 趋势+季节明显时, 平滑法简单高效, 常足够好。
   Strong baseline: with clear trend+season, smoothing is simple, efficient, often enough.

### 下一节 / Next
**14.4 ARIMA / SARIMA / SARIMAX**——经典统计预测的"集大成者"。ARIMA 把**自回归(AR) + 差分(I) + 移动平均(MA)** 结合;SARIMA 加上季节项;SARIMAX 再加外生变量。我们会用 ACF/PACF 定阶, 在航空数据上建模预测。
**14.4 ARIMA / SARIMA / SARIMAX** — the crown jewel of classical forecasting. ARIMA combines **autoregression (AR) + differencing (I) + moving average (MA)**; SARIMA adds seasonal terms; SARIMAX adds exogenous variables. We'll select orders with ACF/PACF and forecast the airline data.
